We can also use this data to predict things. For example, let's load the data for Manhattan, KS from 2014 thorugh 2024:

In [ ]:
import csv
import pandas as pd

years = list(range(2014, 2025))
dataframes = {}

for year in years:
  with open(f"mhk{year}.csv", "r") as file:
    reader = csv.DictReader(file)
    data = list(csv.reader(file, quoting=csv.QUOTE_NONNUMERIC))
    transposed = list(zip(*data))
    dataframes[year] = pd.DataFrame({row[0]: list(row[1:]) for row in transposed})
    dataframes[year]["Year"] = year

# Combine Data Frames
combined_df = pd.concat(dataframes.values(), ignore_index=True)

Then, we can plot the high temperatures for each year!

In [12]:
# Plot high temperatures over the years using Plotly
import plotly.express as px

fig = px.line(combined_df, x="Timestamp", y="AirTemperatureMax", color="Year", title="High Temperatures Over the Years")
fig.show()


Let's graph the data for a single day each year - how about day 100?

In [17]:
# Graph the temperature for day 100 of each year
day_100_df = combined_df[combined_df["Timestamp"] == 100]
fig = px.line(day_100_df, x="Year", y="AirTemperatureMax", title="Maximum Air Temperature for Day 100 of Each Year")
fig.show()

Using a simple linear regression model, we can predict the high temperature for each day of the year in 2025 based on the data from 2014 to 2024. Here's how you can do it using scikit-learn:

In [ ]:
# Perform linear regression for each day of year
from sklearn.linear_model import LinearRegression
import numpy as np

results = {}
slope_100 = 0
for day in range(1, 366):
    day_data = combined_df[combined_df["Timestamp"] == day]
    if len(day_data) > 1:  # Ensure we have enough data points
        X = day_data["Year"].values.reshape(-1, 1)
        y = day_data["AirTemperatureMax"].values
        model = LinearRegression().fit(X, y)
        if day == 100:
            slope_100 = model.coef_[0]  # Store the slope for day 100
        results[day] = model.predict([[2025]])[0]  # Store the predicted value for 2025

# Print results
for day, temp in results.items():
    print(f"Day {day}: Predicted maximum temperature = {temp:.4f} °F")

Day 1: Predicted maximum temperature = 41.1594 °F
Day 2: Predicted maximum temperature = 41.6279 °F
Day 3: Predicted maximum temperature = 48.7182 °F
Day 4: Predicted maximum temperature = 57.7242 °F
Day 5: Predicted maximum temperature = 54.8230 °F
Day 6: Predicted maximum temperature = 50.8739 °F
Day 7: Predicted maximum temperature = 43.3709 °F
Day 8: Predicted maximum temperature = 55.7261 °F
Day 9: Predicted maximum temperature = 52.5400 °F
Day 10: Predicted maximum temperature = 52.5194 °F
Day 11: Predicted maximum temperature = 48.9030 °F
Day 12: Predicted maximum temperature = 41.2491 °F
Day 13: Predicted maximum temperature = 45.9721 °F
Day 14: Predicted maximum temperature = 51.7370 °F
Day 15: Predicted maximum temperature = 37.5497 °F
Day 16: Predicted maximum temperature = 37.5861 °F
Day 17: Predicted maximum temperature = 45.9636 °F
Day 18: Predicted maximum temperature = 45.7752 °F
Day 19: Predicted maximum temperature = 26.3867 °F
Day 20: Predicted maximum temperature = 

Now we can use that model to predict the temperature on that day using the slope of the linear trendline!

In [25]:
# Plot results for day 100 using the predicted value for 2025 and show the trendline using the slope
day_100_df = combined_df[combined_df["Timestamp"] == 100]
# Add the predicted value for 2025 to the DataFrame using concat
predicted_2025 = pd.DataFrame({"Year": [2025], "AirTemperatureMax": [results[100]]})
day_100_df = pd.concat([day_100_df, predicted_2025], ignore_index=True)
fig = px.line(day_100_df, x="Year", y="AirTemperatureMax", title="Maximum Air Temperature for Day 100 of Each Year with Prediction for 2025")
# Add a trendline using the slope from the linear regression as slope_100
slope = slope_100
intercept = day_100_df["AirTemperatureMax"].iloc[0] - slope * day_100_df["Year"].iloc[0]
trendline = slope * day_100_df["Year"] + intercept
fig.add_scatter(x=day_100_df["Year"], y=trendline, mode='lines', name='Trendline')
fig.show()

We can plot the predicted results as well

In [ ]:
# Plot results using plotly and compare to 2025 actual data
# Load 2025 data
with open("mhk2025.csv", "r") as file:
    reader = csv.DictReader(file)
    data = list(csv.reader(file, quoting=csv.QUOTE_NONNUMERIC))
    transposed = list(zip(*data))
    mhk2025 = pd.DataFrame({row[0]: list(row[1:]) for row in transposed})

# Create a DataFrame for plotting
plot_df = pd.DataFrame({
    "Day": list(results.keys()),
    "Predicted": list(results.values()),
    "Actual": mhk2025["AirTemperatureMax"]
})

# Plotting
fig = px.line(plot_df, x="Day", y=["Predicted", "Actual"], title="Predicted vs Actual Maximum Air Temperature for 2025")
fig.show()



Another method would be to perform polynomial regression using the data for an entire year instead of a single day

In [14]:
# Polynomial regression of each year's temperature curve
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

for year in years:
    year_data = combined_df[combined_df["Year"] == year]
    X = year_data["Timestamp"].values.reshape(-1, 1)
    y = year_data["AirTemperatureMax"].values
    model = make_pipeline(PolynomialFeatures(degree=3), LinearRegression())
    model.fit(X, y)
    combined_df.loc[combined_df["Year"] == year, "Predicted"] = model.predict(X)

# Predict 2025 temperatures using the polynomial model
predicted_2025 = model.predict(mhk2025["Timestamp"].values.reshape(-1, 1))
mhk2025["Predicted"] = predicted_2025

# Plotting predicted vs actual for 2025
fig = px.line(mhk2025, x="Timestamp", y=["AirTemperatureMax", "Predicted"], title="Polynomial Regression: Predicted vs Actual Maximum Air Temperature for 2025")
fig.show()